# MyLLM-Tiny Colab

這個 notebook 只負責啟動環境與執行命令；模型邏輯都在 `myllm_tiny/`。請先在 Colab 設定 `Runtime → Change runtime type → T4 GPU`（若可用）。

In [ ]:
!nvidia-smi
!pip install -r requirements.txt

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

PROJECT = '/content/my-llm'
CHECKPOINT_DIR = '/content/drive/MyDrive/MyLLM-Tiny/checkpoints'

## 選擇 tokenizer

預設快速路徑會直接使用現成的 GPT-2 tokenizer；它只下載 tokenizer，不會下載 GPT-2 模型權重。下一個 cell 是可選的，只有要完成原始計畫中的自訓 8K BPE 時才執行。

In [ ]:
%cd $PROJECT
# Optional：若要自訓 8K BPE 才執行這個 cell；使用 GPT-2 tokenizer 可直接跳過。
!mkdir -p artifacts
!python -m myllm_tiny.tokenizer --dataset HuggingFaceFW/fineweb-edu --dataset-config sample-10BT --max-documents 50000 --vocab-size 8192 --output artifacts/tokenizer.json

## Phase 1：1M tokens

若 T4 OOM，將 micro batch 從 8 降到 4 或 2，並相應提高 gradient accumulation。

In [ ]:
%cd $PROJECT
!python -m myllm_tiny.train --pretrained-tokenizer gpt2 --dataset HuggingFaceFW/fineweb-edu --dataset-config sample-10BT --total-tokens 1000000 --micro-batch-size 8 --gradient-accumulation-steps 8 --checkpoint-dir $CHECKPOINT_DIR

## Phase 2：resume 到 5M tokens

下一次 Colab session 或確認 Phase 1 後，才執行這個 cell；將 `--total-tokens` 改成新的累積目標，並指定 `--resume`。

In [ ]:
!python -m myllm_tiny.train --pretrained-tokenizer gpt2 --total-tokens 5000000 --micro-batch-size 8 --gradient-accumulation-steps 8 --checkpoint-dir $CHECKPOINT_DIR --resume $CHECKPOINT_DIR/latest.pt

## Generation

可以在 Phase 1 或 Phase 2 checkpoint 上執行。

In [ ]:
!python -m myllm_tiny.generate --checkpoint $CHECKPOINT_DIR/latest.pt --pretrained-tokenizer gpt2 --prompt 'Artificial intelligence is' --max-new-tokens 80 --temperature 0.8 --top-k 50